# Accessibility info in boolean for Lipas dump

***
Possible way to go not for sure, gave chatgpt our accessibility info and told it to say if the location is accessible or not. Would make it easier for lipas to integrate our data dump trough their API. If no accessibility info from our side, null and if uncertain always False. Results in points_with_accessibility_info checked by hand. 

In [ ]:
from openai import OpenAI
import pandas as pd
from dotenv import load_dotenv
import os

# chatGPT API call

load_dotenv(override=True)
api_key = os.getenv("GPT_KEY")

def is_accessible(text: str) -> bool:
    system_prompt = (
        "You are a classification assistant. "
        "Your job is to decide if a text confirms that a physical location is accessible. "
        "If the text clearly indicates accessibility (e.g., wheelchair ramps, step-free access, etc.), return True. "
        "If it's uncertain, ambiguous, or no info is given, return False."
    )

    user_prompt = f"Text: \"{text}\"\n\nIs this location accessible? Return True or False."


    client = OpenAI(api_key=api_key)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )

    answer = response.choices[0].message.content.strip()
    return answer.lower().startswith("true")

In [ ]:
# our function to give the columns info, accessibility and chall_clas to the api call
# and check each row

def combine_and_check(row):
    if pd.isna(row['accessibil']):
        return None
    access = str(row['accessibil']) if pd.notna(row['accessibil']) else ""
    desc = str(row['info_fi']) if pd.notna(row['info_fi']) else ""
    chall_clas = str(row['chall_clas']) if pd.notna(row['chall_clas']) else ""
    text = f"{desc} {access} {chall_clas}".strip()
    return is_accessible(text)

In [ ]:
# dataset
df = pd.read_csv("points_202506181545.csv")

In [ ]:
pd.reset_option("display.max_rows")
pd.set_option("display.max_columns", None)

# check how many have accessibility info
df[df["accessibil"].notna()]

In [ ]:
# apply for all of those rows that do have accessibility info (we don't need to call the API for rows which don't have)

subset = df[df["accessibil"].notna()]
subset["accessibility_prediction"] = subset.apply(combine_and_check, axis=1)

In [ ]:
# check results by hand because less than 100 rows

pd.set_option("display.max_rows", None)
pd.set_option('display.max_colwidth', None)
subset[["name_fi", "accessibil", "chall_clas", "info_fi", "accessibility_prediction"]]

In [ ]:
# some rows needed changig 

subset.loc[subset['name_fi'] == 'Tummamäen grillikatos', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Rihtniemen leirikeskus', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Mannerlahden leirikeskus', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Luontokapineetti', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Taukopaikka Lahnajärvi', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Kyrön maauimala', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Matikan kuntoradan ja frisbeegolfradan opastuspiste', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Asmandian Info-piste, terassi 10 henkilölle, huussi', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'SF-Saloranta', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Rautilan uimaranta', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Särkijärven uimaranta ja rantasauna', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Kuurnanpään uimapaikka', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Himoisten uimaranta', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Maanpään uimapaikka', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Vähä-Joumon uimapaikka ja sauna', 'accessibility_prediction'] = False
subset.loc[subset['name_fi'] == 'Rautilan rantasauna', 'accessibility_prediction'] = False

In [ ]:
# merge results to original df 

pd.reset_option("display.max_rows")

df = df.merge(
    subset[['gid', 'accessibility_prediction']],
    on='gid',
    how='left'
)


In [ ]:
df_with_accessibility_bool = df[["id", "gid", "name_fi", "class2_fi", "chall_clas", "accessibil", "equipment", "info_fi", "accessibility_prediction"]].copy()

In [ ]:
df_with_accessibility_bool

In [ ]:
df_with_accessibility_bool[["id"]] = df_with_accessibility_bool[["id"]].astype("Int64")

In [ ]:
# save to device

df_with_accessibility_bool.to_csv("points_with_accessibility_bool.csv", index=False)